In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
start_event.record()

In [ ]:
import json
from google.colab import files

input_filename =  '/content/drive/MyDrive/analys_masks.json'
output_filename = '/content/drive/MyDrive/analys_masks/processed_cutouts.json'

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    processed_data = []

    for filename, extracted_text in data.items():
        if extracted_text and len(extracted_text) > 260:
            text_value = None
        else:
            text_value = extracted_text
        processed_data.append({
            "filename": filename,
            "extracted_text": text_value
        })

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(processed_data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(processed_data)} entries.")
    print(f"Saved to '{output_filename}'.")


except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please ensure it is uploaded to the Colab environment.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import json
import re
from google.colab import files

input_filename = '/content/drive/MyDrive/analys_masks/processed_cutouts.json'
output_filename = '/content/drive/MyDrive/analys_masks/processed_with_pn.json'

pattern_6_symbols = re.compile(r'(?:^|\s)([A-Za-z0-9]{6}-\S*)')
pattern_5_symbols = re.compile(r'(?:^|\s)([A-Za-z0-9]{5}-\S*)')
pattern_7_symbols = re.compile(r'(?:^|\s)([A-Za-z0-9]{7}-\S*)')
pattern_8_symbols = re.compile(r'(?:^|\s)([A-Za-z0-9]{8}-\S*)')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        text = item.get("extracted_text")
        item["pn"] = None

        if text:
            match_6 = pattern_6_symbols.search(text)
            if match_6:
                item["pn"] = match_6.group(1)
            else:
                match_5 = pattern_5_symbols.search(text)
                if match_5:
                    item["pn"] = match_5.group(1)
                else:
                    match_7 = pattern_7_symbols.search(text)
                    if match_7:
                        item["pn"] = match_7.group(1)
                    else:
                        match_8 = pattern_8_symbols.search(text)
                        if match_8:
                            item["pn"] = match_8.group(1)
    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} entries.")
    print(f"Saved to '{output_filename}'.")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please upload it to Colab.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import re
from google.colab import files

input_filename = '/content/drive/MyDrive/analys_masks/processed_with_pn.json'
output_filename = '/content/drive/MyDrive/analys_masks/cleaned.json'

clean_pattern = re.compile(r'(?i)s/[nm].*')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        pn = item.get("pn")

        if pn:
            cleaned_pn = clean_pattern.sub('', pn).strip()
            item["pn"] = cleaned_pn if cleaned_pn else None

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items and cleaned the 'pn' fields.")
    print(f"Saved to '{output_filename}'.")



except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please upload it to Colab.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import re

input_filename = '/content/drive/MyDrive/analys_masks/cleaned.json'
output_filename = '/content/drive/MyDrive/analys_masks/filtered.json'
done_filename = '/content/drive/MyDrive/analys_masks/done.json'

exclude_pattern = re.compile(r'^(\d|\d{2}|00\d{2}|000\d)$')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    filtered_data = []
    done_data = []

    for item in data:
        pn = item.get("pn")

        if pn:
            parts = pn.split('-', 1)

            if len(parts) == 2:
                suffix = parts[1]
                if not exclude_pattern.match(suffix):
                    filtered_data.append(item)
                else:
                    done_data.append(item)
            else:
                filtered_data.append(item)
        else:
            done_data.append(item)

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(filtered_data, file, indent=4, ensure_ascii=False)

    with open(done_filename, 'w', encoding='utf-8') as file:
        json.dump(done_data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} total items.")
    print(f" -> Found {len(filtered_data)} items with anomalous suffixes. Saved to '{output_filename}'.")
    print(f" -> Found {len(done_data)} standard items (or missing PNs). Saved to '{done_filename}'.")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please check your Google Drive path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import re

input_filename = '/content/drive/MyDrive/analys_masks/filtered.json'
output_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn.json'

pattern = re.compile(r'(?i)(sn|/n|n|a)')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        pn = item.get("pn")

        item["edited_Pn"] = pn
        if pn and '-' in pn:
            parts = pn.split('-', 1)
            prefix = parts[0]
            suffix = parts[1]
            match = pattern.search(suffix)

            if match:
                seq = match.group(1).upper()
                idx = match.start()

                if seq == 'SN':
                    cleaned_suffix = suffix[:idx]
                else:
                    cut_idx = max(0, idx - 1)
                    cleaned_suffix = suffix[:cut_idx]
                cleaned_suffix = cleaned_suffix.rstrip('-').strip()

                if cleaned_suffix:
                    cleaned_pn = f"{prefix}-{cleaned_suffix}"
                else:
                    cleaned_pn = prefix

                item["edited_Pn"] = cleaned_pn

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items.")
    print(f"Saved the new file to: {output_filename}")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Check that your Google Drive is mounted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import re

input_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn.json'
output_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn_v2.json'

suffix_pattern = re.compile(r'^0+(\d)$')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        edited_pn = item.get("edited_Pn")

        if edited_pn and '-' in edited_pn:
            parts = edited_pn.split('-', 1)
            prefix = parts[0]
            suffix = parts[1]
            match = suffix_pattern.fullmatch(suffix)

            if match:
                last_digit = match.group(1)
                new_suffix = f"000{last_digit}"
                item["edited_Pn"] = f"{prefix}-{new_suffix}"

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items.")
    print(f"Saved the updated file to: {output_filename}")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please verify your Google Drive path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import re
import os

input_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn_v2.json'
done_filename = '/content/drive/MyDrive/analys_masks/done.json'

exclude_pattern = re.compile(r'^(\d|\d{2}|00\d{2}|000\d)$')

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    done_data = []
    if os.path.exists(done_filename):
        with open(done_filename, 'r', encoding='utf-8') as file:
            done_data = json.load(file)

    remaining_data = []
    moved_count = 0

    for item in data:
        edited_pn = item.get("edited_Pn")

        is_done = False

        if edited_pn and '-' in edited_pn:
            parts = edited_pn.split('-', 1)
            suffix = parts[1]

            if exclude_pattern.match(suffix):
                is_done = True

        if is_done:
            done_data.append(item)
            moved_count += 1
        else:
            remaining_data.append(item)

    with open(input_filename, 'w', encoding='utf-8') as file:
        json.dump(remaining_data, file, indent=4, ensure_ascii=False)

    with open(done_filename, 'w', encoding='utf-8') as file:
        json.dump(done_data, file, indent=4, ensure_ascii=False)

    print(f"Success! Evaluated {len(data)} items from the filtered file.")
    print(f" -> Moved {moved_count} items to '{done_filename}'.")
    print(f" -> Kept {len(remaining_data)} items in '{input_filename}'.")
    print(f" -> Total items in 'done.json' is now {len(done_data)}.")

except FileNotFoundError as e:
    print(f"Error: A required file was not found. {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json

input_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn_v2.json'
output_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn_v3.json'

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        pn = item.get("pn")
        edited_pn = item.get("edited_Pn")

        if edited_pn and '-' in edited_pn:
            edited_parts = edited_pn.split('-', 1)
            edited_suffix = edited_parts[1]
            if len(edited_suffix) > 3:
                if pn and '-' in pn:
                    pn_parts = pn.split('-', 1)
                    pn_prefix = pn_parts[0]
                    pn_suffix = pn_parts[1]

                    item["edited_Pn"] = f"{pn_prefix}-{pn_suffix[:1]}"
                    item["edited_Pn1"] = f"{pn_prefix}-{pn_suffix[:2]}"

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items.")
    print(f"Saved the updated file to: {output_filename}")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please verify your Google Drive path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json
import os

input_filename = '/content/drive/MyDrive/analys_masks/filtered_edited_pn_v3.json'
done_filename = '/content/drive/MyDrive/analys_masks/done.json'

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        remaining_data = json.load(file)

    done_data = []
    if os.path.exists(done_filename):
        with open(done_filename, 'r', encoding='utf-8') as file:
            done_data = json.load(file)

    original_done_count = len(done_data)

    done_data.extend(remaining_data)

    with open(done_filename, 'w', encoding='utf-8') as file:
        json.dump(done_data, file, indent=4, ensure_ascii=False)

    print(f"Success! Added {len(remaining_data)} items from the v3 file to 'done.json'.")
    print(f" -> 'done.json' grew from {original_done_count} to {len(done_data)} total items.")

except FileNotFoundError as e:
    print(f"Error: A required file was not found. {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json

input_filename = '/content/drive/MyDrive/analys_masks/done.json'
output_filename = '/content/drive/MyDrive/analys_masks/done1.json'

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        if "edited_Pn" in item:
            item["pn"] = item.pop("edited_Pn")

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items.")
    print(f"Saved the cleaned up file to: {output_filename}")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please verify your Google Drive path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import json

input_filename = '/content/drive/MyDrive/analys_masks/done1.json'
output_filename = '/content/drive/MyDrive/analys_masks/formatted_pns.json'

def clean_raw_ocr(text):
    """Fixes common visual mistakes where the OCR read letters instead of numbers."""
    if not text:
        return text
    text = text.upper()
    text = text.replace('O', '0').replace('T', '7').replace('E', '8')
    text = text.replace('A', '4').replace('G', '6').replace('Z', '2')
    text = text.replace('S', '5').replace('I', '1').replace('B', '8')
    return text

def format_with_zeros(pn_string):
    """Pads the suffix with zeros to ensure it is 4 characters long."""
    if pn_string and '-' in pn_string:
        prefix, suffix = pn_string.split('-', 1)
        padded_suffix = suffix.zfill(4)
        return f"{prefix}-{padded_suffix}"
    return pn_string

try:
    with open(input_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        raw_pn = clean_raw_ocr(item.get("pn"))
        raw_edited_pn1 = clean_raw_ocr(item.get("edited_Pn1"))
        if raw_pn:
            item["formatted_PN"] = format_with_zeros(raw_pn)

        if raw_edited_pn1:
            item["formatted_edited_Pn1"] = format_with_zeros(raw_edited_pn1)

    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"Success! Processed {len(data)} items.")
    print("Applied OCR letter-to-number cleaning and zero-fill formatting.")
    print(f"Saved the updated file to: {output_filename}")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
!pip install rapidfuzz

In [ ]:
import pandas as pd
import json
from rapidfuzz import process, fuzz


excel_path = '/content/drive/MyDrive/analys_masks/Garrett.xlsx'
json_path ='/content/drive/MyDrive/analys_masks/formatted_pns.json'
output_json = '/content/drive/MyDrive/analys_masks/FinalMatchedJson.json'

CONFIDENCE_THRESHOLD = 1.0

# --- Visual Similarity Engine ---
VISUAL_SETS = [
    {'0', '8', 'O', 'B', 'Q', 'D'},
    {'1', '7', 'I', 'L', 'T', '|', '/'},
    {'2', 'Z', '7'},
    {'5', 'S', '6'},
    {'4', 'A', 'H'},
    {'9', 'g', 'q', '0'}
]

def calculate_visual_score(extracted, candidate):
    """
    Custom scorer that applies a tiny penalty for known visual mistakes,
    and a full penalty for completely wrong numbers.
    """
    extracted = str(extracted).upper()
    candidate = str(candidate).upper()

    if len(extracted) != len(candidate):
        return fuzz.ratio(extracted, candidate)

    score = 100.0
    standard_penalty = 100.0 / len(extracted)
    visual_penalty = standard_penalty * 0.2

    for c1, c2 in zip(extracted, candidate):
        if c1 != c2:
            is_visual_mixup = False
            for v_set in VISUAL_SETS:
                if c1 in v_set and c2 in v_set:
                    is_visual_mixup = True
                    break

            if is_visual_mixup:
                score -= visual_penalty
            else:
                score -= standard_penalty

    return max(0.0, score)
from rapidfuzz import process, fuzz, distance

def get_best_match(extracted_str, db_list):
    """
    Finds the best DB match by casting a wide net, then scoring
    based on prefix similarity (more important) and suffix similarity.
    """
    if not extracted_str:
        return None, 0.0

    extracted_str = str(extracted_str).upper()

    top_candidates = process.extract(
        extracted_str,
        db_list,
        scorer=fuzz.ratio,
        limit=30
    )

    best_str = None
    best_score = 0.0

    ext_parts = extracted_str.split('-')
    ext_prefix = ext_parts[0]
    ext_suffix = ext_parts[1] if len(ext_parts) > 1 else ""
    for candidate_str, _, _ in top_candidates:
        cand_str = str(candidate_str).upper()
        cand_parts = cand_str.split('-')
        cand_prefix = cand_parts[0]
        cand_suffix = cand_parts[1] if len(cand_parts) > 1 else ""

        prefix_score = distance.Levenshtein.normalized_similarity(ext_prefix, cand_prefix) * 100

        suffix_score = distance.Levenshtein.normalized_similarity(ext_suffix, cand_suffix) * 100

        combined_score = (prefix_score * 0.75) + (suffix_score * 0.25)

        if combined_score > best_score:
            best_score = combined_score
            best_str = candidate_str

    return best_str, best_score

def get_best_visual_match(extracted_str, db_list):
    """Grabs top 10 standard matches, then re-ranks them visually."""
    if not extracted_str:
        return None, 0.0

    top_candidates = process.extract(
        extracted_str,
        db_list,
        scorer=fuzz.ratio,
        limit=10
    )

    best_str = None
    best_score = 0.0

    for candidate_str, standard_score, _ in top_candidates:
        v_score = calculate_visual_score(extracted_str, candidate_str)
        if v_score > best_score:
            best_score = v_score
            best_str = candidate_str

    return best_str, best_score

def match_part_numbers():
    print(f"Loading database from {excel_path}...")
    try:
        df = pd.read_excel(excel_path, usecols=[0])
        db_pns = df.iloc[:, 0].dropna().astype(str).str.strip().tolist()
        print(f"Loaded {len(db_pns)} part numbers from the database.")
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        return

    print(f"Loading JSON from {json_path}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print("Matching part numbers using Visual Similarity Engine...")
    match_count = 0

    for record in data:
        pn_main = record.get("formatted_PN")
        pn_alt = record.get("formatted_edited_Pn1")

        best_overall_string = None
        best_overall_score = 0.0

        str_main, score_main = get_best_visual_match(pn_main, db_pns)
        if score_main > best_overall_score:
            best_overall_score = score_main
            best_overall_string = str_main

        str_alt, score_alt = get_best_visual_match(pn_alt, db_pns)

        if score_alt >= best_overall_score and str_alt is not None:
            best_overall_score = score_alt
            best_overall_string = str_alt

        if best_overall_score > 0 and best_overall_string is not None:
            record["db_match"] = best_overall_string
            record["match_score"] = round(best_overall_score, 2)
            match_count += 1
        else:
            record["db_match"] = None
            record["match_score"] = 0.0

    print(f"Successfully matched {match_count} out of {len(data)} items.")
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"✅ Matching complete! Saved to {output_json}")

match_part_numbers()

In [ ]:
end_event.record()

torch.cuda.synchronize()

gpu_time_seconds = start_event.elapsed_time(end_event) / 1000.0

print(f"Total GPU Computational Time: {gpu_time_seconds:.2f} seconds")

In [ ]:
import json
import os
import re

gt_json_path = '/content/drive/MyDrive/analys_masks/GarretGT.json'
pred_json_path = '/content/drive/MyDrive/analys_masks/FinalMatchedJson.json'

def extract_img_id(text):
    """Extracts the 'IMG_1234' part from any string to use as a matching key."""
    if not text:
        return None
    match = re.search(r'(IMG_\d+(?:\s\d+)?)', text)
    if match:
        return match.group(1).strip()
    return None

def evaluate_extraction():
    if not os.path.exists(gt_json_path) or not os.path.exists(pred_json_path):
        print("Error: One or both JSON files cannot be found. Check your paths.")
        return

    with open(gt_json_path, 'r', encoding='utf-8') as f:
        raw_gt_data = json.load(f)

    gt_data = {}
    for filepath, data in raw_gt_data.items():
        img_id = extract_img_id(filepath)
        if img_id and "edited_pn" in data:
            gt_data[img_id] = str(data["edited_pn"]).strip()

    with open(pred_json_path, 'r', encoding='utf-8') as f:
        pred_data = json.load(f)

    total_evaluated = 0
    missing_in_gt = 0

    correct_details = []
    mismatch_details = []
    missing_details = []

    for record in pred_data:
        raw_filename = record.get("filename")
        img_id = extract_img_id(raw_filename)

        predicted_db_match = str(record.get("db_match")).strip()
        raw_extracted_pn = str(record.get("formatted_PN", record.get("pn"))).strip()

        match_score = record.get("match_score", 0.0)

        if not img_id or img_id not in gt_data:
            missing_in_gt += 1
            missing_details.append(raw_filename)
            continue

        ground_truth_pn = gt_data[img_id]
        total_evaluated += 1

        row_data = {
            "img_id": img_id,
            "gt": ground_truth_pn,
            "match": predicted_db_match,
            "raw": raw_extracted_pn,
            "score": match_score
        }

        if predicted_db_match == ground_truth_pn:
            correct_details.append(row_data)
        else:
            mismatch_details.append(row_data)

    if total_evaluated == 0:
        print("⚠️ No matching images found. Check your regex or file contents.")
        return

    correct_count = len(correct_details)
    mismatch_count = len(mismatch_details)
    accuracy = (correct_count / total_evaluated) * 100

    print("=== Evaluation Results ===")
    print(f"Total Images Evaluated: {total_evaluated}")
    print(f"Correct DB Matches:     {correct_count}")
    print(f"Incorrect DB Matches:   {mismatch_count}")
    print(f"Missing from GT:        {missing_in_gt}")
    print("-" * 25)
    print(f"Accuracy:               {accuracy:.2f}%\n")

    if missing_details:
        print("=== Missing from Ground Truth ===")
        for filename in missing_details:
            print(f"- {filename}")
        print("\n")

    def print_table(title, data_list):
        if not data_list:
            return
        print(f"=== {title} ===")

        print(f"{'Image ID':<12} | {'Ground Truth':<15} | {'DB Match':<15} | {'Score':<6} | {'Extracted (Formatted)'}")
        print("-" * 85)

        for row in data_list:
            print(f"{row['img_id']:<12} | {row['gt']:<15} | {row['match']:<15} | {str(row['score']):<6} | {row['raw']}")
        print("\n")

    print_table("Correct Matches", correct_details)
    print_table("Mismatches", mismatch_details)

evaluate_extraction()

In [ ]:
end_event.record()
torch.cuda.synchronize()
gpu_time_seconds = start_event.elapsed_time(end_event) / 1000.0

print(f"Total GPU Computational Time: {gpu_time_seconds:.2f} seconds")